# 替代跟踪方法

![AWTT](../../images/alternative_ways_to_trace_0.png)

到目前为止，在本模块中，我们已经了解了 traceable 装饰器，以及如何使用它来设置跟踪。

在本课程中，我们将了解设置跟踪的替代方法，以及何时应该考虑使用这些不同的方法。

## LangChain 和 LangGraph

如果我们使用 LangChain 或 LangGraph，我们需要做的就是设置几个环境变量来设置跟踪

![AWTT](../../images/alternative_ways_to_trace_1.png)

In [ ]:
# 您可以在代码中直接设置
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langsmith-academy"  # 如果您不设置此项，跟踪将进入默认项目

In [ ]:
# 或者您可以使用 .env 文件
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

不要太担心我们这里的图实现，您可以通过我们的 LangGraph Academy 课程了解更多关于 LangGraph 的信息！

In [ ]:
import nest_asyncio
import operator
from langchain.schema import Document
from langchain_core.messages import HumanMessage, AnyMessage, get_buffer_string
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display
from typing import List
from typing_extensions import TypedDict, Annotated
from utils import get_vector_db_retriever, RAG_PROMPT

nest_asyncio.apply()

retriever = get_vector_db_retriever()
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

# 定义图状态
class GraphState(TypedDict):
    question: str
    messages: Annotated[List[AnyMessage], operator.add]
    documents: List[Document]

# 定义节点
def retrieve_documents(state: GraphState):
    messages = state.get("messages", [])
    question = state["question"]
    documents = retriever.invoke(f"{get_buffer_string(messages)} {question}")
    return {"documents": documents}

def generate_response(state: GraphState):
    question = state["question"]
    messages = state["messages"]
    documents = state["documents"]
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    
    rag_prompt_formatted = RAG_PROMPT.format(context=formatted_docs, conversation=messages, question=question)
    generation = llm.invoke([HumanMessage(content=rag_prompt_formatted)])
    return {"documents": documents, "messages": [HumanMessage(question), generation]}

# 定义图
graph_builder = StateGraph(GraphState)
graph_builder.add_node("retrieve_documents", retrieve_documents)
graph_builder.add_node("generate_response", generate_response)
graph_builder.add_edge(START, "retrieve_documents")
graph_builder.add_edge("retrieve_documents", "generate_response")
graph_builder.add_edge("generate_response", END)

simple_rag_graph = graph_builder.compile()
display(Image(simple_rag_graph.get_graph().draw_mermaid_png()))

我们在 LangGraph 中设置了一个简单的图。如果您想了解更多关于 LangGraph 的信息，我强烈建议您查看我们的 LangGraph Academy 课程。

您还可以通过可选的 config 传入元数据或其他字段

In [ ]:
question = "如果我使用 LangChain，如何设置跟踪？"
simple_rag_graph.invoke({"question": question}, config={"metadata": {"foo": "bar"}})

##### 让我们在 LangSmith 中查看一下！

## 跟踪上下文管理器

在 Python 中，您可以使用 trace 上下文管理器将跟踪记录到 LangSmith。这在以下情况下很有用：

您希望为特定代码块记录跟踪。
您希望控制跟踪的输入、输出和其他属性。
使用装饰器或包装器不可行。
以上任一或全部。
上下文管理器与 traceable 装饰器和 wrap_openai 包装器无缝集成，因此您可以在同一应用程序中一起使用它们。

您仍然需要设置您的 `LANGSMITH_API_KEY` 和 `LANGSMITH_TRACING`

![AWTT](../../images/alternative_ways_to_trace_2.png)

In [ ]:
from langsmith import traceable, trace
from openai import OpenAI
from typing import List
import nest_asyncio
from utils import get_vector_db_retriever

MODEL_PROVIDER = "openai"
MODEL_NAME = "gpt-4o-mini"
APP_VERSION = 1.0
RAG_SYSTEM_PROMPT = """您是一个问答任务的助手。
使用以下检索到的上下文片段来回答对话中的最新问题。
如果您不知道答案，请直接说您不知道。
最多使用三句话，保持答案简洁。
"""

openai_client = OpenAI()
nest_asyncio.apply()
retriever = get_vector_db_retriever()

"""
retrieve_documents
- 根据用户的问题从向量存储中返回获取的文档
"""
@traceable
def retrieve_documents(question: str):
    documents = retriever.invoke(question)
    return documents

"""
generate_response
- 在格式化输入后调用 `call_openai` 来生成模型响应
"""
# TODO: 移除 traceable，并使用 with trace()
@traceable
def generate_response(question: str, documents):
    # 注意：我们的文档作为对象列表传入，但我们只想记录一个字符串
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)

    # TODO: 使用 with trace()
    # with trace(
    #     name="生成响应",
    #     run_type="chain", 
    #     inputs={"question": question, "formatted_docs": formatted_docs},
    #     metadata={"foo": "bar"},
    # ) as ls_trace:
    messages = [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"上下文: {formatted_docs} \n\n 问题: {question}"
        }
    ]
    response = call_openai(messages)
    # TODO: 结束您的跟踪并将输出写入 LangSmith
    # ls_trace.end(outputs={"output": response})
    return response

"""
call_openai
- 从 OpenAI 返回聊天完成输出
"""
@traceable
def call_openai(
    messages: List[dict], model: str = MODEL_NAME, temperature: float = 0.0
) -> str:
    response = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response

"""
langsmith_rag
- 调用 `retrieve_documents` 来获取文档
- 调用 `generate_response` 来基于获取的文档生成响应
- 返回模型响应
"""
@traceable
def langsmith_rag(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content

In [ ]:
question = "如何使用跟踪上下文进行跟踪？"
ai_answer = langsmith_rag(question)
print(ai_answer)

## wrap_openai

Python/TypeScript 中的 wrap_openai/wrapOpenAI 方法允许您包装您的 OpenAI 客户端以自动记录跟踪——不需要装饰器或函数包装！包装器与 @traceable 装饰器或 traceable 函数无缝配合，您可以在同一应用程序中使用两者。

您仍然需要设置您的 `LANGSMITH_API_KEY` 和 `LANGSMITH_TRACING`

![AWTT](../../images/alternative_ways_to_trace_3.png)

In [ ]:
# TODO: 导入 wrap_openai
# from langsmith.wrappers import wrap_openai
import openai
from typing import List
import nest_asyncio
from utils import get_vector_db_retriever

MODEL_PROVIDER = "openai"
MODEL_NAME = "gpt-4o-mini"
APP_VERSION = 1.0
RAG_SYSTEM_PROMPT = """您是一个问答任务的助手。
使用以下检索到的上下文片段来回答对话中的最新问题。
如果您不知道答案，请直接说您不知道。
最多使用三句话，保持答案简洁。
"""

# TODO: 包装 OpenAI 客户端
openai_client = openai.Client()

nest_asyncio.apply()
retriever = get_vector_db_retriever()

@traceable(run_type="chain")
def retrieve_documents(question: str):
    return retriever.invoke(question)

@traceable(run_type="chain")
def generate_response(question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    messages = [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"上下文: {formatted_docs} \n\n 问题: {question}"
        }
    ]
    # TODO: 我们不再需要在嵌套函数调用上使用 @traceable，
    # wrap_openai 为我们处理这个
    return call_openai(messages)

@traceable
def call_openai(
    messages: List[dict],
) -> str:
    return openai_client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
    )

@traceable(run_type="chain")
def langsmith_rag_with_wrap_openai(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content

In [ ]:
question = "如何使用 wrap_openai 进行跟踪？"
ai_answer = langsmith_rag_with_wrap_openai(question)
print(ai_answer)

包装的 OpenAI 客户端接受与 @traceable 装饰函数相同的 langsmith_extra 参数

In [ ]:
messages = [
    {
        "role": "user",
        "content": "天空是什么颜色的？"
    }
]

openai_client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    langsmith_extra={"metadata": {"foo": "bar"}},
)

## [高级] RunTree

另一种更明确的向 LangSmith 记录跟踪的方法是通过 RunTree API。此 API 允许您更好地控制跟踪 - 您可以手动创建运行和子运行来组装您的跟踪。您仍然需要设置您的 `LANGSMITH_API_KEY`，但这种方法不需要 `LANGSMITH_TRACING`。

![AWTT](../../images/alternative_ways_to_trace_4.png)

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_PROJECT"] = "langsmith-academy"

In [ ]:
from dotenv import load_dotenv
# 我的环境变量定义在 .env 文件中
load_dotenv(dotenv_path="../../.env", override=True)

让我们将 `LANGSMITH_TRACING` 设置为 false，因为在这种情况下我们使用 RunTree 手动创建运行。

In [ ]:
import os
os.environ["LANGSMITH_TRACING"] = "false"

from langsmith import utils
utils.tracing_is_enabled() # 这应该返回 false

我们重写了我们的 RAG 应用程序，除了这次我们通过函数调用传递 RunTree 参数，并在每个层创建子运行。这为我们的 RunTree 提供了与使用 @traceable 自动建立的相同层次结构

In [ ]:
from langsmith import RunTree
from openai import OpenAI
from typing import List
import nest_asyncio
from utils import get_vector_db_retriever

openai_client = OpenAI()
nest_asyncio.apply()
retriever = get_vector_db_retriever()

def retrieve_documents(parent_run: RunTree, question: str):
    # 创建子运行
    child_run = parent_run.create_child(
        name="检索文档",
        run_type="retriever",
        inputs={"question": question},
    )
    documents = retriever.invoke(question)
    # 发布我们子运行的输出
    child_run.end(outputs={"documents": documents})
    child_run.post()
    return documents

def generate_response(parent_run: RunTree, question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    rag_system_prompt = """您是一个问答任务的助手。
    使用以下检索到的上下文片段来回答对话中的最新问题。
    如果您不知道答案，请直接说您不知道。
    最多使用三句话，保持答案简洁。
    """
    # 创建子运行
    child_run = parent_run.create_child(
        name="生成响应",
        run_type="chain",
        inputs={"question": question, "documents": documents},
    )
    messages = [
        {
            "role": "system",
            "content": rag_system_prompt
        },
        {
            "role": "user",
            "content": f"上下文: {formatted_docs} \n\n 问题: {question}"
        }
    ]
    openai_response = call_openai(child_run, messages)
    # 发布我们子运行的输出
    child_run.end(outputs={"openai_response": openai_response})
    child_run.post()
    return openai_response

def call_openai(
    parent_run: RunTree, messages: List[dict], model: str = "gpt-4o-mini", temperature: float = 0.0
) -> str:
    # 创建子运行
    child_run = parent_run.create_child(
        name="OpenAI 调用",
        run_type="llm",
        inputs={"messages": messages},
    )
    openai_response = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    # 发布我们子运行的输出
    child_run.end(outputs={"openai_response": openai_response})
    child_run.post()
    return openai_response

def langsmith_rag(question: str):
    # 创建根 RunTree
    root_run_tree = RunTree(
        name="聊天管道",
        run_type="chain",
        inputs={"question": question}
    )

    # 将我们的 RunTree 传递到嵌套函数调用中
    documents = retrieve_documents(root_run_tree, question)
    response = generate_response(root_run_tree, question, documents)
    output = response.choices[0].message.content

    # 发布我们的最终输出
    root_run_tree.end(outputs={"generation": output})
    root_run_tree.post()
    return output

In [ ]:
question = "如何使用 RunTree 进行跟踪？"
ai_answer = langsmith_rag(question)
print(ai_answer)